# BridgeRisk: Exploratory Analysis & One-Year-Ahead Condition Prediction

**Author:** Divyansh Kumar Singh (DKS) · M.Tech Civil Engineering (Hydraulic), IIT Kanpur  
**GitHub:** [DKS-MANAGER](https://github.com/DKS-MANAGER)  
**Domain:** Bridge Asset Management / Infrastructure Deterioration Modeling  

---

## 1. Civil Engineering Context & Objective
Highway bridge decks deteriorate progressively under seasonal thermal cycling, moisture intrusion, de-icing salts, and cyclic heavy vehicle loading. In the United States, the Federal Highway Administration (FHWA) National Bridge Inventory (NBI) records periodic condition ratings on a 0–9 scale:
- **7–9:** Good condition
- **5–6:** Fair condition
- **0–4:** Poor condition (deficiencies requiring major rehabilitation, load restriction, or replacement)

### Research Question
Can we predict whether a bridge deck will fall into **Poor condition (Rating $\le$ 4) in year $t+1$** using inspection records from year $t$?  
We analyze multi-state inventory data from **Maine (ME)**, **Hawaii (HI)**, and **Delaware (DE)** spanning **2021–2025** using a strict chronological split (2021–2024 train transitions, 2024–2025 out-of-time test).

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, classification_report

# Setup plot styles
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 10

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "reports"
FIGURES_DIR = BASE_DIR / "figures"
MODELS_DIR = BASE_DIR / "models"

print(f"Project root: {BASE_DIR}")

## 2. Dataset Overview: Multi-State NBI Inventory
We load the chronological training transitions (2021→2024) and the out-of-time test dataset (2024→2025).

In [ ]:
train_df = pd.read_parquet(DATA_DIR / "train_2021_2024.parquet")
test_df = pd.read_parquet(DATA_DIR / "test_2024_2025.parquet")

print(f"Training set records (2021-2024 transitions): {len(train_df):,d}")
print(f"Testing set bridges (2024-2025 out-of-time):    {len(test_df):,d}")

print("\n--- Test Set Distribution by State ---")
if "state" in test_df.columns:
    print(test_df["state"].value_counts())

y_test = test_df["target_deck_poor_next_year"]
print(f"\nTest Positive Prevalence (Deck <= 4 in 2025): {y_test.sum():,d} / {len(y_test):,d} ({y_test.mean():.2%})")

## 3. Exploratory Analysis: Condition Rating & Bridge Age
Let us examine how bridge deck ratings and structure age are distributed across the bridge inventory.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Deck condition ratings
deck_ratings = pd.to_numeric(test_df["DECK_COND_058"], errors="coerce").dropna()
sns.countplot(x=deck_ratings.astype(int), ax=axes[0], color="#2b5c8f")
axes[0].set_title("Distribution of Current Deck Condition (NBI Item 58)")
axes[0].set_xlabel("Condition Rating (0-9 Scale)")
axes[0].set_ylabel("Number of Bridges")
axes[0].axvline(4.5, color="red", linestyle="--", label="Poor Threshold (<= 4)")
axes[0].legend()

# 2. Bridge age distribution
ages = test_df["bridge_age"].dropna()
axes[1].hist(ages, bins=30, color="#5c82a6", edgecolor="white")
axes[1].set_title("Bridge Age Distribution (Years Since Construction)")
axes[1].set_xlabel("Age (Years)")
axes[1].set_ylabel("Number of Bridges")
axes[1].axvline(ages.median(), color="darkblue", linestyle="--", label=f"Median Age: {ages.median():.0f} yrs")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Model Evaluation on 2024→2025 Out-of-Time Test Set
We compare the Majority Baseline, Logistic Regression, Random Forest, and XGBoost models on unseen test bridges.

In [ ]:
metrics_df = pd.read_csv(REPORTS_DIR / "model_metrics.csv")
print(metrics_df[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "false_negatives", "false_negative_rate"]].to_string(index=False))

## 5. SHAP Feature Attribution & Civil Engineering Interpretation
SHAP explains the contribution of each bridge attribute to the model's prediction.

In [ ]:
shap_df = pd.read_csv(REPORTS_DIR / "shap_feature_importance.csv")
print("Top 10 Feature Drivers of Deck Deterioration:")
print(shap_df.head(10).to_string(index=False))

plt.figure(figsize=(9, 5))
top10 = shap_df.head(10).iloc[::-1]
labels = [f.replace("num__", "").replace("cat__", "") for f in top10["feature"]]
plt.barh(range(len(top10)), top10["mean_abs_shap"], color="#2b5c8f")
plt.yticks(range(len(top10)), labels)
plt.xlabel("Mean |SHAP Value| (Impact on Model Output)")
plt.title("Top 10 Feature Drivers of One-Year-Ahead Deck Deterioration")
plt.tight_layout()
plt.show()

## 6. Maintenance Prioritization Shortlist
Raw probabilities are weighted by traffic demand, current condition severity, and scour criticality to establish inspection and maintenance shortlists:
$$\text{Priority Score} = P(\text{Poor}) \times M_{\text{condition}} \times M_{\text{traffic}} \times M_{\text{scour}}$$

In [ ]:
priority_df = pd.read_csv(REPORTS_DIR / "maintenance_priority_2025.csv")
print("--- Maintenance Priority Breakdown (4,558 Bridges) ---")
print(priority_df["priority_class"].value_counts())

print("\n--- Top 10 Highest Priority Bridges ---")
cols = ["bridge_id", "state", "current_deck_condition", "average_daily_traffic", "scour_criticality", "priority_score", "priority_class"]
available = [c for c in cols if c in priority_df.columns]
print(priority_df[available].head(10).to_string(index=False))